# Aula 10 — Como Funciona um LLM

Três exercícios, um depois de cada microaula. Rode as células na ordem.
Onde aparecer `____`, escreva o que se pede e rode: são experimentos com
um modelo de verdade, e quase nenhum tem resposta única.

**Travou?** [Colas dos exercícios](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios).

## Preparação

🧰 Baixa as ferramentas do curso (o tokenizador da Aula 9, as funções
que desenham e medem) e o modelo do professor, treinado por uma hora em
Machado de Assis. Leva uns 20 segundos.

In [ ]:
import os
import shutil
import urllib.request

if os.path.exists("../../data/llm_curso.py"):          # só no PC do professor
    shutil.copy("../../data/llm_curso.py", "llm_curso.py")
else:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/klein-natan/"
                               "nanodegree-AI-Atitus/main/data/llm_curso.py", "llm_curso.py")
from llm_curso import *

professor = modelo_do_professor()
print("Pronto.")

---
## Exercício 1: o token vira vetor

📝 Antes: [quiz da Microaula 1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-1-o-token-vira-vetor).
🆘 Travou? [Cola do Exercício 10.1](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-1)

🎯 Ver como um texto vira exemplos de treino, e como o sentido das
palavras virou geometria dentro do modelo.

### 1.a Uma frase sua vira exemplos de treino

Escreva uma frase qualquer. A célula mostra os tokens e os exemplos de
treino que saem dela: em cada um, o modelo lê o começo e tem que prever
o token seguinte.

In [ ]:
frase = "____"          # uma frase sua, entre as aspas
quantos = exemplos_de_treino(frase)

In [ ]:
if "____" in frase or not frase.strip():
    print("❌ Escreva uma frase entre as aspas, no lugar do ____.")
else:
    print(f"✅ {quantos + 1} tokens viraram {quantos} exemplos de treino, todos na mesma passada. "
          "O livro inteiro vira 1,3 milhão deles.")

✏️ Por que o último token da frase não vira um exemplo? →

### 1.b Palpite: quem fica perto de quem?

No modelo treinado, cada token é um vetor de 128 números (o embedding).
Antes de rodar, dê um palpite: qual palavra o modelo pôs mais perto de
"elle"?

In [ ]:
palpite = "____"        # uma palavra, entre as aspas
vizinhos(professor, "elle", palpite)

Agora troque `"quando"` por outras palavras e rode de novo: `"era"`,
`"dous"`, `"olhos"`, `"mãe"`.

In [ ]:
vizinhos(professor, "quando")

✏️ Os vizinhos são sinônimos, ou palavras que fazem o mesmo papel na
frase? →

### 1.c Um grupo seu no mapa 3D

🧰 `mapa_3d` desenha cinco grupos de palavras (números, pessoas, verbos,
partes do corpo, conectivos), espremendo os 128 números em 3, e mostra o
espaço de três lados. Acrescente um grupo seu: quatro palavras do mesmo
tipo. Palavras raras viram vários tokens e ficam de fora (a célula avisa).

In [ ]:
meu_grupo = ["____", "____", "____", "____"]
mapa_3d(professor, meu_grupo)

✏️ O seu grupo ficou junto? Perto de qual dos outros? →

---
## Exercício 2: a atenção

📝 Antes: [quiz da Microaula 2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-2-a-atencao).
🆘 Travou? [Cola do Exercício 10.2](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-2)

🎯 Ver para onde o modelo olha, e pôr à prova a máscara e a posição.

### 2.a Para onde um substantivo olha

🧰 `para_onde_olha` mostra, em cada uma das quatro camadas, quanto uma
palavra olha para cada token que veio antes dela (a média das quatro
cabeças).

In [ ]:
para_onde_olha(professor, "A casa de Capitú tinha uma janela para a rua.", "janela")

Na camada 4, `janela` olha principalmente para `uma`: o substantivo
achou o artigo dele, que diz se a palavra é masculina ou feminina. Vale
para outras frases? Escreva uma frase sua com "um" ou "uma" antes de um
substantivo, e diga qual é o substantivo.

In [ ]:
frase = "____"          # uma frase com "um" ou "uma" antes de um substantivo
palavra = "____"        # o substantivo
para_onde_olha(professor, frase, palavra)

✏️ Na camada 4, o seu substantivo olhou para o artigo? →

### 2.b A máscara: o futuro não muda o passado

🧰 `apostas` mostra, em cada posição, o token em que o modelo aposta para
vir em seguida. Troque só a última palavra da frase e compare as duas
tabelas.

In [ ]:
antes = apostas(professor, "O menino era pai do homem.")
print()
depois = apostas(professor, "O menino era pai do ____.")    # troque só a última palavra

In [ ]:
comeco = len(codificar("O menino era pai do"))
if "____" in str(depois) or len(depois) <= comeco:
    print("❌ Escreva uma palavra no lugar do ____.")
elif antes[:comeco] == depois[:comeco]:
    print("✅ Até o 'do', as apostas são idênticas: nenhuma posição olhou para a palavra que "
          "vem depois dela. É a máscara causal.")
else:
    print("❌ Troque só a última palavra, sem mexer no começo da frase.")

### 2.c A ordem importa

🧰 `estranheza` mede quanto o modelo se surpreende com um texto (é a
perda): quanto menor, mais provável ele acha o texto. Embaralhe a frase,
usando exatamente as mesmas palavras.

In [ ]:
certa = "Não sei se a senhora sabe que elle não veio hontem."
embaralhada = "____"    # as mesmas palavras, em outra ordem

print(f"certa:       {estranheza(professor, certa):.2f}")
print(f"embaralhada: {estranheza(professor, embaralhada):.2f}")

In [ ]:
if sorted(certa.split()) != sorted(embaralhada.split()):
    print("❌ Use exatamente as mesmas palavras da frase certa, só em outra ordem.")
elif estranheza(professor, embaralhada) > estranheza(professor, certa):
    print("✅ Mesmas palavras, outra ordem, e o modelo estranha muito mais. Ele sabe onde cada "
          "palavra está por causa do embedding de posição; sem ele, as duas dariam o mesmo número.")
else:
    print("🤔 Embaralhe mais: troque várias palavras de lugar.")

---
## Exercício 3: o modelo inteiro

📝 Antes: [quiz da Microaula 3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/aula-10-como-funciona-um-llm#microaula-3-o-bloco-e-o-modelo-inteiro).
🆘 Travou? [Cola do Exercício 10.3](https://klein-natan.github.io/nanodegree-AI-Atitus/#/llm-do-zero/colas-dos-exercicios#exercicio-10-3)

🎯 O LLM inteiro em vinte linhas de Keras, e a diferença que o treino faz.

### 3.a O modelo, em Keras

Leia a célula com calma antes de rodar: é o modelo inteiro, e cada linha
é uma peça da microaula.

In [ ]:
def montar_modelo(camadas=4, dimensao=128, cabecas=4):
    entrada = keras.Input(shape=(None,), dtype="int32")
    x = TokenEPosicao(dimensao, name="embedding")(entrada)          # token + posição
    for i in range(camadas):                                         # os blocos
        h = keras.layers.LayerNormalization(name=f"norma_a{i}")(x)
        h = keras.layers.MultiHeadAttention(num_heads=cabecas, key_dim=dimensao // cabecas,
                                            name=f"atencao{i}")(h, h, use_causal_mask=True)
        x = x + h                                                    # atalho
        h = keras.layers.LayerNormalization(name=f"norma_b{i}")(x)
        h = keras.layers.Dense(2 * dimensao, activation="gelu", name=f"densa{i}")(h)
        h = keras.layers.Dense(dimensao, name=f"volta{i}")(h)
        x = x + h                                                    # atalho
    x = keras.layers.LayerNormalization(name="norma_final")(x)
    saida = keras.layers.Dense(1024, name="saida")(x)                # 1.024 logits por posição
    return keras.Model(entrada, saida)


modelo = montar_modelo()
contar_pesos(modelo)

### 3.b Do tamanho de um GPT-2

O GPT-2 pequeno, de 2019, tem 12 blocos, dimensão 768 e 12 cabeças. Monte
um modelo desse tamanho, com o nosso vocabulário, e compare.

In [ ]:
grande = montar_modelo(camadas=____, dimensao=____, cabecas=12)
total = contar_pesos(grande)
print(f"\n{total / 809_728:.0f} vezes o nosso modelo")

In [ ]:
if total > 50_000_000:
    print("✅ Mesmas peças, mesmo código: mudaram três números. É isso que separa o nosso "
          "modelo de um GPT-2, e, com mais zeros, de um ChatGPT.")
else:
    print("❌ O GPT-2 pequeno tem camadas=12 e dimensao=768.")

✏️ No modelo grande, que peça ficou com a maior parte dos pesos? →

### 3.c Antes e depois do treino

🧰 O `modelo` que você montou tem pesos sorteados. A célula monta outro
igual e carrega nele os pesos do professor: mesma arquitetura, outros
números. Compare os dois.

In [ ]:
treinado = montar_modelo()
treinado.load_weights(baixar_pesos_do_professor())

machado = "Não sei se a senhora sabe que elle não veio hontem."
for nome, m in [("pesos sorteados", modelo), ("pesos treinados", treinado)]:
    print(f"=== {nome}: estranheza {estranheza(m, machado):.2f}")
    chances(m, "A casa de")
    print(gerar(m, "A casa de", 40), "\n")

O modelo sorteado dá estranheza perto de 6,93, que é ln(1.024): ele chuta
igual entre os 1.024 tokens. O treinado escolhe entre poucas dezenas.

### 3.d Complete a frase

Escreva o começo de uma frase e veja as apostas do modelo treinado e uma
continuação sorteada.

In [ ]:
comeco = "____"         # o começo de uma frase
chances(treinado, comeco)
print(gerar(treinado, comeco))

✏️ Ache um começo em que o favorito passa de 50%, e outro em que nenhum
passa de 10%. O que diferencia os dois? →

---
## O que você viu hoje

| Experimento | O que ele mostra |
|---|---|
| uma frase vira exemplos | cada posição prevê a seguinte, tudo na mesma passada |
| vizinhos e mapa 3D | palavras que fazem o mesmo papel têm vetores parecidos |
| para onde olha | a atenção liga palavras, como o substantivo ao artigo |
| trocar a última palavra | a máscara: o futuro não muda o passado |
| embaralhar | a posição: a ordem importa para o modelo |
| `montar_modelo` | o LLM inteiro, e o GPT-2 com os mesmos tijolos |
| sorteado e treinado | toda a diferença está nos números dos pesos |